In [1]:
!pip install nltk spacy

import nltk
import pandas as pd
import numpy as np
import string

In [2]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [3]:
import pandas as pd

url = '/content/synthetic_resume_dataset.csv'

df = pd.read_csv(url)

df.head()


,resume_text,job_role
0,Worked with stakeholders to define KPIs and me...,Business Analyst
1,Managed social media and influencer outreach. ...,Marketing Manager
2,Designed and executed digital marketing strate...,Marketing Manager
3,"Skilled in SQL, Tableau, and process modeling....",Business Analyst
4,Optimized database queries and application per...,Backend Developer


In [4]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word.isalpha() and word not in stop_words]
    return " ".join(tokens)

df['clean_text'] = df['resume_text'].apply(preprocess)
df[['clean_text', 'job_role']].head()


,clean_text,job_role
0,worked stakeholder define kpis metric metric d...,Business Analyst
1,managed social medium influencer outreach infl...,Marketing Manager
2,designed executed digital marketing strategy s...,Marketing Manager
3,skilled sql tableau process modeling modeling ...,Business Analyst
4,optimized database query application performan...,Backend Developer


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=1000)
X = vectorizer.fit_transform(df['clean_text'])
y = df['job_role']


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [7]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)


LogisticRegression(max_iter=1000)

In [8]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Classification Report:
                     precision    recall  f1-score   support

         Accountant       1.00      1.00      1.00        15
  Backend Developer       1.00      1.00      1.00         9
   Business Analyst       1.00      1.00      1.00        15
       Data Analyst       1.00      1.00      1.00         7
     Data Scientist       1.00      1.00      1.00         9
 Frontend Developer       1.00      1.00      1.00        12
       HR Executive       1.00      1.00      1.00         9
  Marketing Manager       1.00      1.00      1.00         7
Mechanical Engineer       1.00      1.00      1.00         8
 Software Developer       1.00      1.00      1.00         9

           accuracy                           1.00       100
          macro avg       1.00      1.00      1.00       100
       weighted avg       1.00      1.00      1.00       100

Confusion Matrix:
[[15  0  0  0  0  0  0  0  0  0]
 [ 0  9  0  0  0  0  0  0  0  0]
 [ 0  0 15  0  0  0  0  0  0  0]
 [ 

In [9]:
test_resume = """Skilled in Java, Spring Boot, and MySQL. Developed REST APIs for ecommerce application."""
clean_test = preprocess(test_resume)
vectorized_test = vectorizer.transform([clean_test])
predicted_role = model.predict(vectorized_test)[0]

print("Predicted Job Role:", predicted_role)


Predicted Job Role: Backend Developer


In [10]:
!pip install PyMuPDF
import fitz  # PyMuPDF


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 60.3 MB/s eta 0:00:00


In [11]:
from google.colab import files

uploaded_file = files.upload()  # Upload your resume file here
file_name = next(iter(uploaded_file))


Saving Udhaya_M_Resume.pdf to Udhaya_M_Resume.pdf


In [12]:
def extract_text(file_path):
    if file_path.endswith('.pdf'):
        doc = fitz.open(file_path)
        text = ""
        for page in doc:
            text += page.get_text()
        return text
    elif file_path.endswith('.txt'):
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    else:
        return ""

resume_raw_text = extract_text(file_name)
print(resume_raw_text[:500])  # Preview first 500 characters


Udhaya M
Python Programmer | AI&ML developer
CONTACT
Chennai,Tamil Nadu
+91 8778724176
udhayaud2208@gmail.com
LinkedIn
GitHub
SKILLS
ABOUT
B.Tech Student at the intersection of Artificial Intelligence and Data Science with
expertise in Python, Data Analysis and Machine Learning. Passionate about automation
and rapid prototyping , thrives on working with the latest as well as proven AI frameworks
and tools to turn concepts into products. Experienced at developing AI concepts fo real
world applica


In [13]:
cleaned_resume = preprocess(resume_raw_text)
vectorized_resume = vectorizer.transform([cleaned_resume])
predicted_job = model.predict(vectorized_resume)[0]

print("✅ Predicted Job Role:", predicted_job)


✅ Predicted Job Role: Data Scientist


In [14]:
import joblib
joblib.dump(model, "resume_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")


['tfidf_vectorizer.pkl']